# AeroNetra — Video Tracking & Unique Vehicle Counts (Phase 3)

**Purpose:** drive a detector frame by frame through a tracker so each vehicle keeps one identity, then report the *unique* vehicle count rather than a per-image box count.

**Kaggle inputs**
1. `aeronetra-trained-weights` — output of notebook 02 (or the COCO `yolo11n.pt` for a smoke test).
2. `aeronetra-video-clip` — a short clip (`.mp4`). Upload one as a private Kaggle dataset; a phone or dashcam clip works for checking the loop.

**Tracker:** ByteTrack by default. Switch to `botsort.yaml` for better accuracy on UAV footage (it compensates for camera motion) at roughly four times the runtime.

Self-contained: the tracking helpers below are inlined from `src/aeronetra/tracking/ops.py`, so no `pip install aeronetra` is needed.

In [ ]:
# ============================================================
# Cell 1: Setup
# ============================================================
!pip install -q ultralytics

import json
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# ============================================================
# Cell 2: Data structures (inlined from aeronetra.detection.types)
# ============================================================

@dataclass
class BoundingBox:
    xmin: float
    ymin: float
    xmax: float
    ymax: float

    @property
    def xyxy(self):
        return (self.xmin, self.ymin, self.xmax, self.ymax)


@dataclass
class Detection:
    box: BoundingBox
    class_id: int
    class_name: str
    confidence: float


@dataclass
class TrackedDetection:
    """A detection plus the identity the tracker assigned to it."""
    detection: Detection
    track_id: int
    frame_index: int = 0

In [ ]:
# ============================================================
# Cell 3: Tracking helpers (inlined from aeronetra.tracking.ops)
# ============================================================

def track_class(detections):
    """The class a whole track is reported as: its most frequent class id."""
    counts = Counter(d.class_id for d in detections)
    return min(counts, key=lambda cid: (-counts[cid], cid))


def summarize(frames, min_len=1, names=None):
    """Counts unique vehicles; min_len drops tracks too short to be traffic."""
    names = names or {}
    per_track = defaultdict(list)
    for frame in frames:
        for det in frame:
            per_track[det.track_id].append(det)
    kept = [dets for dets in per_track.values() if len(dets) >= min_len]
    per_class = Counter()
    for dets in kept:
        cid = track_class(dets)
        per_class[names.get(cid, str(cid))] += 1
    return {
        "frames": len(frames),
        "unique": len(kept),
        "raw_ids": len(per_track),
        "dropped": len(per_track) - len(kept),
        "per_class": dict(sorted(per_class.items())),
    }


def color_for(track_id):
    """A stable colour per track id, so the same vehicle keeps one colour."""
    hue = int((track_id * 137) % 180)
    blue, green, red = cv2.cvtColor(
        np.uint8([[[hue, 200, 255]]]), cv2.COLOR_HSV2BGR
    )[0, 0]
    return int(blue), int(green), int(red)


def draw_tracks(image, tracked):
    for item in tracked:
        x1, y1, x2, y2 = (int(round(v)) for v in item.detection.box.xyxy)
        color = color_for(item.track_id)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            image, f"ID {item.track_id}", (x1, max(y1 - 6, 12)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA,
        )
    return image

In [ ]:
# ============================================================
# Cell 4: Inputs and settings
# ============================================================
INPUT_ROOT = Path("/kaggle/input")
CLIP_SLUG = "aeronetra-video-clip"
WEIGHTS_SLUG = "aeronetra-trained-weights"

clip_root = INPUT_ROOT / CLIP_SLUG
weights_root = INPUT_ROOT / WEIGHTS_SLUG
clips = sorted(clip_root.rglob("*.mp4")) if clip_root.exists() else []
weights = sorted(weights_root.rglob("*.pt")) if weights_root.exists() else []

assert clips, f"No .mp4 found under {clip_root} — attach your clip dataset."
assert weights, f"No .pt found under {weights_root} — attach your weights dataset."

VIDEO_PATH = clips[0]
WEIGHTS_PATH = weights[0]

# VisDrone class order, as produced by notebook 02. If you use COCO weights
# for a smoke test, replace this with the COCO ids you care about.
CLASS_NAMES = {
    0: "car", 1: "van", 2: "truck", 3: "tricycle",
    4: "awning-tricycle", 5: "bus", 6: "motor", 7: "bicycle",
}

TRACKER = "bytetrack.yaml"   # or "botsort.yaml" (more accurate, ~4x slower)
CONF, IOU = 0.25, 0.45
MIN_TRACK_LENGTH = 5         # frames a track must appear in to count
MAX_FRAMES = None            # set an int for a quick look at a long clip

print("clip    :", VIDEO_PATH)
print("weights :", WEIGHTS_PATH)
print("tracker :", TRACKER)

In [ ]:
# ============================================================
# Cell 5: Track the clip and write an annotated copy
# ============================================================
from ultralytics import YOLO

model = YOLO(str(WEIGHTS_PATH))

capture = cv2.VideoCapture(str(VIDEO_PATH))
assert capture.isOpened(), f"Could not open {VIDEO_PATH}"

fps = capture.get(cv2.CAP_PROP_FPS) or 25.0
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
out_path = Path("/kaggle/working/tracked.mp4")
writer = cv2.VideoWriter(
    str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
)
assert writer.isOpened(), "Could not open the video writer"

frames = []
seen_ids = set()
index = 0
try:
    while MAX_FRAMES is None or index < MAX_FRAMES:
        ok, frame = capture.read()
        if not ok:
            break

        # persist=True is what makes an identity survive between frames.
        result = model.track(
            frame, persist=True, tracker=TRACKER, conf=CONF, iou=IOU, verbose=False
        )[0]

        tracked = []
        if result.boxes is not None and result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy()
            track_ids = result.boxes.id.cpu().numpy()
            for box, conf, cls_id, track_id in zip(boxes, confs, classes, track_ids):
                cid = int(cls_id)
                tracked.append(TrackedDetection(
                    detection=Detection(
                        box=BoundingBox(*(float(v) for v in box)),
                        class_id=cid,
                        class_name=CLASS_NAMES.get(cid, str(cid)),
                        confidence=float(conf),
                    ),
                    track_id=int(track_id),
                    frame_index=index,
                ))

        for item in tracked:
            seen_ids.add(item.track_id)
        frames.append(tracked)

        annotated = draw_tracks(frame.copy(), tracked)
        cv2.putText(
            annotated, f"unique: {len(seen_ids)}", (12, 28),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2, cv2.LINE_AA,
        )
        writer.write(annotated)
        index += 1
finally:
    capture.release()
    writer.release()

summary = summarize(frames, min_len=MIN_TRACK_LENGTH, names=CLASS_NAMES)
print(json.dumps(summary, indent=2))
print("annotated video:", out_path)

## Reading the result

`unique` is the number of distinct identities that survived the minimum-length filter — that is the vehicle count. `raw_ids` is how many identities the tracker created in total, so `dropped` is the flicker: short-lived ids that were almost certainly false positives rather than vehicles.

If `dropped` is a large share of `raw_ids`, the detector is noisy at this confidence and raising `CONF` (or `MIN_TRACK_LENGTH`) will clean the count up. If two boxes on one vehicle carry different ids, the tracker is fragmenting a track — that is the case `track_buffer` and `botsort.yaml` exist to address.

The annotated video is the check: the count in the corner should only rise when a genuinely new vehicle appears, never while one is merely occluded.